# Mistral-7B QLoRA on 12 GB with offloading

This notebook uses 4-bit loading, QLoRA prep, gradient checkpointing, and CPU/disk offloading to make fine-tuning more likely to fit on a 12 GB GPU.

In [ ]:
import os
import gc
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.cuda.empty_cache()
gc.collect()

model_name = "mistralai/Mistral-7B-v0.1"
cache_dir = "./models"
offload_dir = "./offload"
output_dir = "./output"

os.makedirs(cache_dir, exist_ok=True)
os.makedirs(offload_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Let Accelerate place as much as possible on GPU and spill the rest to CPU/disk.
max_memory = {
    0: "10GiB",
    "cpu": "48GiB",
}

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=offload_dir,
    offload_state_dict=True,
    torch_dtype=torch.float16,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ds = load_dataset("tatsu-lab/alpaca")

def tokenize_function(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        inp = inp if inp else ""
        text = f"Instruction: {inst}\nInput: {inp}\nOutput: {out}"
        texts.append(text)

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=64,
    )
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

tokenized_ds = ds.map(
    tokenize_function,
    batched=True,
    remove_columns=["instruction", "input", "output", "text"],
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    optim="paged_adamw_8bit",
    max_steps=50,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    data_collator=data_collator,
)

trainer.train()
